In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
from tqdm import tqdm  # This gives you a nice progress bar
from loading_module_custom import get_dataset_path

# ---------------- CONFIG ----------------
input_file = get_dataset_path("preprocessed")
print(f"Input file: {input_file}")
output_file = get_dataset_path("processed")
print(f"Output file: {output_file}")
# ----------------------------------------

# 1. Load the dataset
# We don't use chunks here so we can access stations quickly in memory
ds = xr.open_dataset(input_file)



# ------ Check if all stations have unique lat/lon/code values ------
# Pick any two stations (first two in the dataset)
station1, station2 = ds.station.values[:2]

# Print their code, lat, and lon
for s in [station1, station2]:
    print(f"Station: {s}")
    print(f"  Code: {ds.code.sel(station=s).item()}")
    print(f"  Lat : {ds.lat.sel(station=s).item()}")
    print(f"  Lon : {ds.lon.sel(station=s).item()}")
    print()

stations = ds.station.values


Input file: /Users/arnavdangmali/MainFolder/University/Projects/ICRAR-Weather-Forcasting/DPIRD_final_stations.nc
Output file: /Users/arnavdangmali/MainFolder/University/Projects/ICRAR-Weather-Forcasting/DPIRD_final_stations_interpolated.nc
Station: Floreat Park
  Code: FL
  Lat : -31.951181
  Lon : 115.792334

Station: Jerdacuttup
  Code: JP001
  Lat : -33.77361
  Lon : 120.45667



In [ ]:
# 2. Define the new 5-minute time axis for the full range
new_time = pd.date_range(start=ds.time.min().values, 
                         end=ds.time.max().values, 
                         freq='5min')

# 3. Get list of all stations
stations = ds.station.values
all_interpolated_stations = []

ds


<xarray.Dataset> Size: 3GB
Dimensions:                       (station: 192, time: 105248)
Coordinates:
  * station                       (station) <U22 17kB 'Floreat Park' ... 'Gut...
  * time                          (time) datetime64[ns] 842kB 2022-01-01T08:0...
    lat                           (station) float64 2kB ...
    lon                           (station) float64 2kB ...
    code                          (station) <U5 4kB ...
Data variables: (12/18)
    airTemperature                (station, time) float64 162MB ...
    apparentAirTemperature        (station, time) float64 162MB ...
    relativeHumidity              (station, time) float64 162MB ...
    dewPoint                      (station, time) float64 162MB ...
    panEvaporation                (station, time) float64 162MB ...
    evapotranspiration_shortCrop  (station, time) float64 162MB ...
    ...                            ...
    frostCondition                (station, time) float64 162MB ...
    heatCondition                 (station, time) float64 162MB ...
    wind_3m_speed                 (station, time) float64 162MB ...
    wind_3m_degN                  (station, time) float64 162MB ...
    wind_10m_speed                (station, time) float64 162MB ...
    wind_10m_degN                 (station, time) float64 162MB ...
Attributes:
    time_zone:      UTC+08:00
    time_standard:  local_time
    comment:        All timestamps are local time in UTC+08:00 (Australia/Per...

In [ ]:
print(f"Starting interpolation for {len(stations)} stations...")

# 4. Loop through stations (Safe for Memory)
for st in tqdm(stations):
    # Select one station
    ds_st = ds.sel(station=st)
    
    # We use a dictionary to store interpolated variables for this specific station
    interp_vars = {}
    
    for var in ds_st.data_vars:
        if 'time' in ds_st[var].dims:
            # Drop NaNs, Resample to 5min, and use PCHIP
            # .to_series() is very fast on M3 for single-station slices
            series = ds_st[var].to_series().dropna()
            
            if not series.empty:
                # The actual math happens here
                series_interp = series.resample("5min").asfreq().interpolate(method='pchip')
                
                # Convert back to DataArray
                interp_vars[var] = xr.DataArray(
                    series_interp.values,
                    coords={"time": series_interp.index},
                    dims=["time"],
                    name=var,
                    attrs=ds_st[var].attrs
                )
    
    # Create a station-specific dataset and add the station coordinate back
    if interp_vars:
        ds_st_interp = xr.Dataset(interp_vars)
        ds_st_interp = ds_st_interp.assign_coords(station=st)
        all_interpolated_stations.append(ds_st_interp)

# 5. Combine all stations back together
print("Merging all stations...")
ds_final = xr.concat(all_interpolated_stations, dim="station")

# 6. Save to disk
print(f"Saving to {output_file}...")
ds_final.to_netcdf(output_file)

print("Done!")

Starting full-year interpolation for Allanooka...
  -> Processing airTemperature...
  -> Processing apparentAirTemperature...
  -> Processing relativeHumidity...
  -> Processing dewPoint...
  -> Processing panEvaporation...
  -> Processing evapotranspiration_shortCrop...
  -> Processing evapotranspiration_tallCrop...
  -> Processing richardsonUnits...
  -> Processing solarExposure...
  -> Processing rainfall...
  -> Processing deltaT...
  -> Processing wetBulb...
  -> Processing frostCondition...
  -> Processing heatCondition...
  -> Processing wind_3m_speed...
  -> Processing wind_3m_degN...
  -> Processing wind_10m_speed...
  -> Processing wind_10m_degN...

Success! Images saved to the 'interpolation_previews' folder.


In [ ]:
# Load both (use your actual filenames)
ds_new = xr.open_dataset(get_dataset_path("processed"))

# 1. Compare Time Steps
orig_time_len = len(ds.time)
new_time_len = len(ds_new.time)

# 2. Compare Total Grid Points (Stations * Time Steps)
# This represents every potential coordinate where a temperature or rain value exists
num_stations = len(ds.station)
total_orig = orig_time_len * num_stations
total_new = new_time_len * num_stations

print(f"{'Metric':<25} | {'Original (15m)':<15} | {'New (5m)':<15}")
print("-" * 60)
print(f"{'Time Steps':<25} | {orig_time_len:<15,} | {new_time_len:<15,}")
print(f"{'Stations':<25} | {num_stations:<15} | {num_stations:<15}")
print(f"{'Total Grid Points':<25} | {total_orig:<15,} | {total_new:<15,}")
print("-" * 60)
print(f"Your dataset has grown by {new_time_len / orig_time_len:.1f}x in resolution.")

Metric                    | Original (15m)  | New (5m)       
------------------------------------------------------------
Time Steps                | 105,248         | 315,742        
Stations                  | 185             | 185            
Total Grid Points         | 19,470,880      | 58,412,270     
------------------------------------------------------------
Your dataset has grown by 3.0x in resolution.


In [ ]:
# Add coords from all_station_coordinates.csv to final dataset Final_stations_interpolated.nc per 

# Read station coordinates
coords_df = pd.read_csv(get_dataset_path("dpird_station_list"))
print("Station Coordinates DataFrame:")
print(coords_df.head())

# print all names of stations in ds
print("Stations in dataset:")
print(ds.station.values)    

# iterate through coords_df and check if all stations are in ds
missing_stations = []
for station in coords_df['name']:
    if station not in ds.station.values:
        missing_stations.append(station)
if missing_stations:
    print("Missing stations in dataset:")
    print(missing_stations)
else:
    print("All stations from coordinates CSV are present in the dataset.")


# print all the coords currently associated with the stations
# print("Current coordinates in dataset:")
# print(ds.coords)

# ds = ds.merge(coords_df.set_index('station').to_xarray())
# ds.to_netcdf("Final_stations_interpolated_with_coords.nc")
# print("Added coordinates to Final_stations_interpolated.nc and saved as Final_stations_interpolated_with_coords.nc")

# ds






Station Coordinates DataFrame:
    code         name        lat         lon    alt
0  AN001    Allanooka -29.063612  114.997161  131.0
1  AM001       Amelup -34.270827  118.268523  200.0
2  SH002      Babakin -32.125480  118.004060  313.0
3     BA  Badgingarra -30.338049  115.539491  284.0
4  BP001     Balingup -33.796200  116.063980  227.0
Stations in dataset:
['Floreat Park' 'Jerdacuttup' 'Moora' 'Mount Howick' 'Mingenew NW'
 'Cascade NW' 'Dudawa' 'Broome (Skuthorpe)' 'Kojaneerup South' 'Tincurrin'
 'Trayning West' 'Eneabba' 'Manypeaks' 'Katanning GRDC' 'Dinninup'
 'Allanooka' 'Burracoppin South' 'South Perth' 'Denmark' 'Coomalbidgup'
 'Jerramungup' 'Northam' 'Lancelin East' 'Mingenew' 'New Norcia' 'Binnu'
 'Frankland' 'Holt Rock' 'Morawa' 'Mount Barker South' 'Koorda' 'Nannup'
 'Yanmah' 'Donnybrook' 'Yuna NE' 'Corrigin' 'Esperance Downs'
 'Warradarge East' 'Canna East' 'Corrigin East' 'Boyatup' 'Cascade NE'
 'Wilyabrup' 'Frankland North' 'Belka East' 'Magenta' 'Babakin'
 'Dumbleyung